# process_court_infocuria_files_v4

This notebook preserves the **InfoCuria ingestion, output folders, clean manifests, and Excel outputs**, while using the Curia-style candidate extraction engine.

Core behavior:

- Smart HTML and plain-text decoding with scored encodings.
- Scored HTML content-container selection.
- Three inexpensive native PDF candidates:
  - PyMuPDF sorted text
  - PyMuPDF sorted blocks
  - `pdftotext -layout`
- Conditional MinerU OCR fallback.
- MinerU replaces native extraction only when it materially improves quality and passes safeguards.
- Separate readable and regex/LLM text outputs.
- Clean-file and candidate-level CSV/XLSX manifests.
- Source signature checks prevent stale output reuse.


## 1. Configuration

In [1]:
from __future__ import annotations

import os
import re
import sys
import json
import time
import shutil
import hashlib
import tempfile
import subprocess
import unicodedata
from collections import Counter
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, Any, List, Tuple

import pandas as pd
from tqdm.auto import tqdm

try:
    import fitz  # PyMuPDF
except Exception:
    fitz = None

try:
    from bs4 import BeautifulSoup, UnicodeDammit
except Exception:
    BeautifulSoup = None
    UnicodeDammit = None

try:
    from charset_normalizer import from_bytes as charset_from_bytes
except Exception:
    charset_from_bytes = None

try:
    from ftfy import fix_text as ftfy_fix_text
except Exception:
    ftfy_fix_text = None

# ---------------------------------------------------------------------
# Project root and InfoCuria folders
# ---------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd().resolve()

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "output").exists() or (candidate / "data").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
# PROJECT_ROOT = Path("/home/edik/projects/eccjeu").resolve()  # optional override

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output" / "court_infocuria"
EXCEL_DIR = OUTPUT_DIR / "excel"

PROCESSED_DIR = DATA_DIR / "processed" / "court_infocuria"
TEXT_DIR = PROCESSED_DIR
MARKDOWN_DIR = PROCESSED_DIR
ERROR_DIR = PROCESSED_DIR / "errors"
CANDIDATE_DATA_DIR = PROCESSED_DIR / "candidates"

DOWNLOAD_MANIFEST_STEM = "infocuria_document_download_manifest"
DOWNLOAD_MANIFEST_CSV = EXCEL_DIR / f"{DOWNLOAD_MANIFEST_STEM}.csv"
DOWNLOAD_MANIFEST_XLSX = EXCEL_DIR / f"{DOWNLOAD_MANIFEST_STEM}.xlsx"

CLEAN_FILE_MANIFEST_PATH = EXCEL_DIR / "infocuria_clean_file_manifest.csv"
CLEAN_FILE_MANIFEST_XLSX = EXCEL_DIR / "infocuria_clean_file_manifest.xlsx"
CANDIDATE_MANIFEST_PATH = EXCEL_DIR / "infocuria_extraction_candidate_manifest.csv"
CANDIDATE_MANIFEST_XLSX = EXCEL_DIR / "infocuria_extraction_candidate_manifest.xlsx"

# Backward-compatible names used in inspection cells.
CLEAN_MANIFEST_CSV = CLEAN_FILE_MANIFEST_PATH
CLEAN_MANIFEST_XLSX = CLEAN_FILE_MANIFEST_XLSX

for folder in [PROCESSED_DIR, ERROR_DIR, CANDIDATE_DATA_DIR, EXCEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Run behavior
# ---------------------------------------------------------------------
EXTRACTION_VERSION = 4
FORCE_REPROCESS = True
PROCESS_LIMIT = None       # e.g. 20 for a test run
SAVE_EVERY = 100

# Save all candidate text variants only when debugging. Candidate metadata is
# always written to CSV/XLSX.
SAVE_ALL_CANDIDATE_TEXTS = False
SAVE_REVIEW_CANDIDATE_TEXTS = True

# ---------------------------------------------------------------------
# Native PDF and OCR behavior
# ---------------------------------------------------------------------
RUN_PDFTOTEXT = True
PDFTOTEXT_CLI = shutil.which("pdftotext")

RUN_MINERU = True
RUN_MINERU_FOR_ALL_PDFS = False
MINERU_TIMEOUT_SECONDS = 1200
MINERU_CLI = shutil.which("mineru") or shutil.which("magic-pdf")

MINERU_COMMAND_TEMPLATE = (
    [
        MINERU_CLI,
        "-p", "{input_pdf}",
        "-o", "{output_dir}",
        "-m", "ocr",
        "-b", "pipeline",
    ]
    if MINERU_CLI
    else []
)

# Curia-style conservative OCR thresholds.
MIN_CHARS_ANY = 200
MIN_PDF_CHARS_GOOD = 800
MIN_SCORE_ACCEPTABLE = 35.0
MIN_SCORE_OK = 55.0
MIN_NATIVE_SCORE_TO_RUN_OCR = 35.0
MIN_MINERU_SCORE_GAIN = 2.5
METHOD_TIE_TOLERANCE = 0.5
MIN_MINERU_NATIVE_LENGTH_RATIO = 0.60

SERIOUS_OCR_REASONS = {
    "very_short_text",
    "replacement_characters",
    "possible_mojibake",
    "private_use_characters",
    "repeated_character_garbage",
    "low_language_plausibility",
}

TEXT_EXTENSIONS = {".txt", ".text"}
HTML_EXTENSIONS = {".html", ".htm", ".xhtml"}
PDF_EXTENSIONS = {".pdf"}
SUPPORTED_EXTENSIONS = TEXT_EXTENSIONS | HTML_EXTENSIONS | PDF_EXTENSIONS

print("Python:", sys.version)
print("Project root:", PROJECT_ROOT)
print("InfoCuria download manifest:", DOWNLOAD_MANIFEST_CSV)
print("Processed folder:", PROCESSED_DIR)
print("Clean manifest:", CLEAN_FILE_MANIFEST_PATH)
print("Candidate manifest:", CANDIDATE_MANIFEST_PATH)
print("PyMuPDF available:", fitz is not None)
print("BeautifulSoup available:", BeautifulSoup is not None)
print("charset-normalizer available:", charset_from_bytes is not None)
print("ftfy available:", ftfy_fix_text is not None)
print("pdftotext:", PDFTOTEXT_CLI)
print("MinerU:", MINERU_CLI)

Python: 3.12.3 (main, Jun 19 2026, 12:46:00) [GCC 13.3.0]
Project root: /home/edik/projects/eccjeu
InfoCuria download manifest: /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_document_download_manifest.csv
Processed folder: /home/edik/projects/eccjeu/data/processed/court_infocuria
Clean manifest: /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_clean_file_manifest.csv
Candidate manifest: /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_extraction_candidate_manifest.csv
PyMuPDF available: True
BeautifulSoup available: True
charset-normalizer available: True
ftfy available: True
pdftotext: /usr/bin/pdftotext
MinerU: /home/edik/projects/.venv/bin/mineru


## 2. Load and normalize the InfoCuria download manifest

In [2]:
PATH_COLUMNS = [
    "final_local_path", "local_path", "existing_local_path", "intended_local_path",
    "downloaded_file_path", "download_file_path", "file_path", "raw_file_path",
    "saved_path", "target_filename", "target_path", "download_path",
]

CASE_COLUMNS = [
    "case_number_clean", "case_number_raw", "case_number", "publishedId",
    "procNumber", "procedure_key", "procedure_key_base",
    "infocuria_slash_procedure_key",
]

DOC_ID_COLUMNS = [
    "document_dedup_key", "docId", "logicDocId", "jpLogicDocId", "publishedId",
    "idPublished", "idProcedure", "procedureId", "affId",
]

DOCTYPE_COLUMNS = ["normalized_docTypeCode", "docTypeCode", "docType", "document_type"]
DATE_COLUMNS = ["docDate", "date", "document_date"]
LANG_COLUMNS = ["selected_language", "final_success_language", "bestLang", "language"]
TITLE_COLUMNS = ["case_name", "title", "document_title", "name"]
URL_COLUMNS = ["source_url", "document_url", "url", "download_url"]
DOCUMENT_KEY_COLUMNS = ["document_dedup_key", "document_key", "logicDocId", "jpLogicDocId"]


def read_manifest(path_csv: Path, path_xlsx: Path) -> pd.DataFrame:
    if path_csv.exists():
        print(f"Reading CSV manifest: {path_csv}")
        frame = pd.read_csv(path_csv, low_memory=False)
        frame["__download_manifest_path"] = str(path_csv)
        return frame
    if path_xlsx.exists():
        print(f"Reading Excel manifest: {path_xlsx}")
        frame = pd.read_excel(path_xlsx)
        frame["__download_manifest_path"] = str(path_xlsx)
        return frame
    raise FileNotFoundError(
        "Could not find InfoCuria document download manifest. Expected one of:\n"
        f"  {path_csv}\n"
        f"  {path_xlsx}"
    )


def first_nonempty(row: pd.Series, columns: List[str]) -> str:
    for column in columns:
        if column not in row.index:
            continue
        value = row.get(column)
        if pd.isna(value):
            continue
        text = str(value).strip()
        if text and text.lower() not in {"nan", "none", "null"}:
            return text
    return ""


def resolve_path(value: str) -> Optional[Path]:
    if not value or str(value).strip().lower() in {"nan", "none", "null"}:
        return None

    raw_values = [part.strip() for part in re.split(r"[;|]", str(value)) if part.strip()]
    candidates: List[Path] = []

    for raw in raw_values:
        if raw.startswith(("http://", "https://")):
            continue
        path = Path(raw)
        if path.is_absolute():
            candidates.append(path)
        else:
            candidates.extend([
                PROJECT_ROOT / path,
                DATA_DIR / path,
                OUTPUT_DIR / path,
                EXCEL_DIR / path,
                Path.cwd() / path,
            ])

    for candidate in candidates:
        if candidate.exists() and candidate.is_file():
            return candidate.resolve()
    return None


def infer_file_format(path: Optional[Path]) -> str:
    if path is None:
        return ""
    extension = path.suffix.lower()
    if extension in PDF_EXTENSIONS:
        return "pdf"
    if extension in HTML_EXTENSIONS:
        return "html"
    if extension in TEXT_EXTENSIONS:
        return "txt"
    return extension.lstrip(".") or "unknown"


def safe_id(value: str, fallback: str = "missing") -> str:
    text = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value or "").strip())
    text = re.sub(r"_+", "_", text).strip("_.-")
    return text or fallback


def source_signature(path: Optional[Path]) -> Tuple[Optional[int], Optional[int]]:
    if path is None or not path.exists():
        return None, None
    stat = path.stat()
    return int(stat.st_size), int(stat.st_mtime_ns)


def build_document_id(row: pd.Series, source_row_i: int, raw_path: Optional[Path]) -> str:
    preferred = first_nonempty(row, DOC_ID_COLUMNS)
    if preferred:
        return preferred

    pieces = [
        first_nonempty(row, CASE_COLUMNS),
        first_nonempty(row, DOCTYPE_COLUMNS),
        first_nonempty(row, DATE_COLUMNS),
        first_nonempty(row, LANG_COLUMNS),
        raw_path.name if raw_path else "",
        str(source_row_i),
    ]
    digest = hashlib.sha1("|".join(pieces).encode("utf-8", errors="ignore")).hexdigest()[:16]
    return f"infocuria_{digest}"


def prepare_work_manifest(frame: pd.DataFrame) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    for source_row_i, source_row in frame.iterrows():
        raw_path = resolve_path(first_nonempty(source_row, PATH_COLUMNS))
        file_format = infer_file_format(raw_path)
        eligible = bool(
            raw_path
            and raw_path.exists()
            and raw_path.suffix.lower() in SUPPORTED_EXTENSIONS
        )

        if raw_path is None:
            skip_reason = "missing_or_unresolved_file_path"
        elif raw_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            skip_reason = "unsupported_file_type"
        else:
            skip_reason = ""

        document_id = build_document_id(source_row, source_row_i, raw_path)
        size_bytes, modified_ns = source_signature(raw_path)

        rows.append({
            "source_row_i": source_row_i,
            "document_id": document_id,
            "document_key": first_nonempty(source_row, DOCUMENT_KEY_COLUMNS),
            "case_number": first_nonempty(source_row, CASE_COLUMNS),
            "case_number_clean": first_nonempty(source_row, CASE_COLUMNS),
            "case_name": first_nonempty(source_row, TITLE_COLUMNS),
            "document_type": first_nonempty(source_row, DOCTYPE_COLUMNS),
            "document_date": first_nonempty(source_row, DATE_COLUMNS),
            "language": first_nonempty(source_row, LANG_COLUMNS),
            "source_url": first_nonempty(source_row, URL_COLUMNS),
            "downloaded_file_path": str(raw_path) if raw_path else "",
            "raw_file_path": str(raw_path) if raw_path else "",
            "file_exists": bool(raw_path and raw_path.exists()),
            "file_type": file_format,
            "file_format": file_format,
            "source_size_bytes": size_bytes,
            "source_modified_ns": modified_ns,
            "processing_eligible": eligible,
            "processing_skip_reason": skip_reason,
            "__download_manifest_path": source_row.get("__download_manifest_path", ""),
        })

    work = pd.DataFrame(rows)

    duplicate_mask = work["document_id"].astype(str).duplicated(keep=False)
    if duplicate_mask.any():
        # Preserve all documents instead of allowing output overwrites.
        duplicate_counts: Dict[str, int] = {}
        repaired_ids = []
        for _, row in work.iterrows():
            base = str(row["document_id"])
            if not duplicate_mask.loc[row.name]:
                repaired_ids.append(base)
                continue
            duplicate_counts[base] = duplicate_counts.get(base, 0) + 1
            repaired_ids.append(f"{base}__dup{duplicate_counts[base]:03d}")
        work["document_id"] = repaired_ids

    assert work["document_id"].notna().all()
    assert not work["document_id"].astype(str).duplicated().any()
    return work


manifest_df = read_manifest(DOWNLOAD_MANIFEST_CSV, DOWNLOAD_MANIFEST_XLSX)
print("Rows in download manifest:", len(manifest_df))
print("Columns:", list(manifest_df.columns))

work_df = prepare_work_manifest(manifest_df)
print("Eligible files:", int(work_df["processing_eligible"].sum()))
print(work_df["file_format"].value_counts(dropna=False))
display(work_df.head(20))

Reading Excel manifest: /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_document_download_manifest.xlsx
Rows in download manifest: 77085
Columns: ['internal_key', 'procedure_key', 'procedure_key_base', 'case_number_raw', 'case_number_clean', 'affId', 'procedureId', 'publishedId', 'docId', 'logicDocId', 'jpLogicDocId', 'idProcedure', 'idPublished', 'docType', 'docTypeCode', 'normalized_docTypeCode', 'docDate', 'available_languages', 'docFormats', 'candidate_years', 'download_url', 'selected_language', 'selected_format', 'selected_year', 'target_filename', 'intended_local_path', 'existing_local_path', 'final_local_path', 'file_present_on_disk', 'file_exists_before_download', 'download_success', 'download_status', 'attempted_urls', 'attempted_http_statuses', 'attempt_count', 'working_url', 'working_language', 'working_format', 'working_year', 'http_status', 'error', 'bytes', '__download_manifest_path']
Eligible files: 77085
file_format
html    62280
pdf     14805
Name: c

,source_row_i,document_id,document_key,case_number,case_number_clean,case_name,document_type,document_date,language,source_url,downloaded_file_path,raw_file_path,file_exists,file_type,file_format,source_size_bytes,source_modified_ns,processing_eligible,processing_skip_reason,__download_manifest_path
0,0,1216021,id_85906,C-1/00,C-1/00,,ARRET_SOM,2001-12-13,EN,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/projects/eccjeu/data/raw/court_info...,True,pdf,pdf,136051,1783029449540399943,True,,/home/edik/projects/eccjeu/output/court_infocu...
1,1,616977,id_46950,C-1/00,C-1/00,,ARRET,2001-12-13,EN,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/projects/eccjeu/data/raw/court_info...,True,html,html,93964,1782738505956166056,True,,/home/edik/projects/eccjeu/output/court_infocu...
2,2,616472,id_46421,C-1/00 SA,C-1/00 SA,,ORD,2001-05-29,EN,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/projects/eccjeu/data/raw/court_info...,True,html,html,11133,1782738506051488046,True,,/home/edik/projects/eccjeu/output/court_infocu...
3,3,1216138,id_85947,C-1/00 SA,C-1/00 SA,,ORD_SOM,2001-05-29,EN,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/projects/eccjeu/data/raw/court_info...,True,pdf,pdf,62340,1783029450399645570,True,,/home/edik/projects/eccjeu/output/court_infocu...
4,4,1258909,id_85572,C-1/01 P,C-1/01 P,,ORD_SOM,2001-09-20,EN,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/projects/eccjeu/data/raw/court_info...,True,pdf,pdf,70066,1782220628500000000,True,,/home/edik/projects/eccjeu/output/court_infocu...
5,5,616689,id_46652,C-1/01 P,C-1/01 P,,ORD,2001-09-20,EN,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/projects/eccjeu/data/raw/court_info...,True,html,html,34447,1782220592750000000,True,,/home/edik/projects/eccjeu/output/court_infocu...
6,6,618967,id_49061,C-1/02,C-1/02,,ARRET,2004-04-01,EN,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/projects/eccjeu/data/raw/court_info...,True,html,html,24382,1782738506172571286,True,,/home/edik/projects/eccjeu/output/court_infocu...
7,7,701344,id_84842,C-1/02,C-1/02,,ORD_COMM,2003-08-22,EN,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/projects/eccjeu/data/raw/court_info...,True,html,html,1687,1783344055277684719,True,,/home/edik/projects/eccjeu/output/court_infocu...
8,8,624317,id_54478,C-1/02,C-1/02,,ORD_COMM,2003-09-06,FR,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/projects/eccjeu/data/raw/court_info...,True,html,html,1936,1783344056189457735,True,,/home/edik/projects/eccjeu/output/court_infocu...
9,9,625082,id_55247,C-1/02,C-1/02,,ARRET_SOM,2004-04-01,EN,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/projects/eccjeu/data/raw/court_info...,True,html,html,2601,1783029451238959763,True,,/home/edik/projects/eccjeu/output/court_infocu...


## 3. Encoding, cleaning, and quality scoring

In [3]:
MOJIBAKE_PATTERNS = [
    "Ã", "Â", "â€™", "â€œ", "â€", "ðŸ", "�",
]

LEGAL_TERMS = {
    "en": [
        "commission", "decision", "court", "article", "applicant", "undertaking",
        "competition", "judgment", "order", "appeal", "action", "annulment",
        "proceedings", "costs", "admissible", "inadmissible",
    ],
    "de": [
        "kommission", "entscheidung", "gericht", "artikel", "kläger", "unternehmen",
        "wettbewerb", "urteil", "beschluss", "rechtsmittel", "klage", "nichtigkeit",
        "verfahren", "kosten", "zulässig", "unzulässig",
    ],
    "fr": [
        "commission", "décision", "cour", "article", "requérant", "entreprise",
        "concurrence", "arrêt", "ordonnance", "recours", "annulation",
        "procédure", "dépens", "recevable", "irrecevable",
    ],
    "it": [
        "commissione", "decisione", "corte", "articolo", "ricorrente", "impresa",
        "concorrenza", "sentenza", "ordinanza", "ricorso", "annullamento",
        "procedimento", "spese", "ricevibile", "irricevibile",
    ],
}

COMMON_WORDS = {
    "en": ["the", "of", "and", "to", "in", "that", "for", "on", "with"],
    "de": ["der", "die", "das", "und", "von", "zu", "in", "für", "mit"],
    "fr": ["de", "la", "le", "et", "des", "les", "du", "pour", "dans"],
    "it": ["di", "la", "il", "e", "del", "della", "per", "in", "con"],
}


def read_file_bytes(path: Path) -> bytes:
    return path.read_bytes()


def normalize_unicode(text: str, form: str = "NFKC") -> str:
    text = unicodedata.normalize(form, text or "")
    replacements = {
        "\u00a0": " ",
        "\u200b": "",
        "\ufeff": "",
        "\r\n": "\n",
        "\r": "\n",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def score_decoded_text(text: str) -> float:
    if not text:
        return -100.0
    n = len(text)
    replacement_ratio = text.count("�") / max(n, 1)
    control_ratio = sum(
        1 for ch in text
        if unicodedata.category(ch).startswith("C") and ch not in "\n\t"
    ) / max(n, 1)
    mojibake_ratio = sum(text.count(p) for p in MOJIBAKE_PATTERNS) / max(n, 1)
    printable_ratio = sum(ch.isprintable() or ch in "\n\t" for ch in text) / max(n, 1)
    return (
        printable_ratio * 30
        - replacement_ratio * 300
        - control_ratio * 200
        - mojibake_ratio * 180
    )


def extract_declared_html_encoding(data: bytes) -> Optional[str]:
    head = data[:8192]
    ascii_head = head.decode("ascii", errors="ignore")
    patterns = [
        r'<meta[^>]+charset\s*=\s*["\']?\s*([A-Za-z0-9._-]+)',
        r'<meta[^>]+content\s*=\s*["\'][^"\']*charset\s*=\s*([A-Za-z0-9._-]+)',
        r'<\?xml[^>]+encoding\s*=\s*["\']([^"\']+)',
    ]
    for pattern in patterns:
        match = re.search(pattern, ascii_head, flags=re.I)
        if match:
            return match.group(1).strip()
    return None


def decode_bytes_smart(data: bytes, is_html: bool = False) -> Tuple[str, Dict[str, Any]]:
    candidates: List[Tuple[str, str, str]] = []

    if data.startswith(b"\xef\xbb\xbf"):
        candidates.append(("utf-8-sig", "bom", data.decode("utf-8-sig", errors="replace")))
    elif data.startswith((b"\xff\xfe", b"\xfe\xff")):
        candidates.append(("utf-16", "bom", data.decode("utf-16", errors="replace")))

    declared = extract_declared_html_encoding(data) if is_html else None
    if declared:
        try:
            candidates.append((declared, "declared", data.decode(declared, errors="replace")))
        except Exception:
            pass

    if is_html and UnicodeDammit is not None:
        try:
            dammit = UnicodeDammit(data, is_html=True, smart_quotes_to=None)
            if dammit.unicode_markup:
                candidates.append((
                    dammit.original_encoding or "unknown",
                    "unicode_dammit",
                    dammit.unicode_markup,
                ))
        except Exception:
            pass

    try:
        candidates.append(("utf-8", "strict_utf8", data.decode("utf-8", errors="strict")))
    except UnicodeDecodeError:
        pass

    if charset_from_bytes is not None:
        try:
            matches = charset_from_bytes(data)
            for match in list(matches)[:4]:
                text = str(match)
                candidates.append((
                    getattr(match, "encoding", None) or "unknown",
                    "charset_normalizer",
                    text,
                ))
        except Exception:
            pass

    for encoding in ["cp1252", "latin-1"]:
        try:
            candidates.append((encoding, "fallback", data.decode(encoding, errors="replace")))
        except Exception:
            pass

    if not candidates:
        candidates.append(("utf-8", "last_resort", data.decode("utf-8", errors="replace")))

    unique = {}
    for enc, source, text in candidates:
        key = hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()
        unique.setdefault(key, (enc, source, text))

    scored = []
    for enc, source, text in unique.values():
        base_score = score_decoded_text(text)
        repaired = None
        repaired_score = None

        if ftfy_fix_text is not None:
            try:
                repaired = ftfy_fix_text(text)
                repaired_score = score_decoded_text(repaired)
            except Exception:
                repaired = None

        use_repaired = (
            repaired is not None
            and repaired != text
            and repaired_score is not None
            and repaired_score > base_score + 0.5
        )
        final_text = repaired if use_repaired else text
        final_score = repaired_score if use_repaired else base_score

        scored.append({
            "encoding": enc,
            "encoding_source": source,
            "text": final_text,
            "decode_score": float(final_score),
            "ftfy_applied": bool(use_repaired),
            "declared_encoding": declared or "",
        })

    best = max(scored, key=lambda x: x["decode_score"])
    meta = {k: v for k, v in best.items() if k != "text"}
    meta["decode_candidate_count"] = len(scored)
    return normalize_unicode(best["text"]), meta


def clean_text_readable(text: str) -> str:
    text = normalize_unicode(text, form="NFC")
    text = text.replace("\u00ad", "")
    text = re.sub(
        r"(?<=[A-Za-zÀ-ÖØ-öø-ÿ])[-‐‑]\s*\n\s*(?=[a-zà-öø-ÿ])",
        "",
        text,
    )
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip() + "\n" if text.strip() else ""


def clean_text_for_regex_and_llm(text: str) -> str:
    text = normalize_unicode(text, form="NFKC")
    text = clean_text_readable(text)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip() + "\n" if text.strip() else ""


def repeated_character_ratio(text: str) -> float:
    if not text:
        return 1.0
    repeated = sum(len(m.group(0)) for m in re.finditer(r"(.)\1{5,}", text, flags=re.S))
    return repeated / max(len(text), 1)


def duplicate_line_ratio(text: str) -> float:
    lines = [
        re.sub(r"\s+", " ", line).strip().lower()
        for line in text.splitlines()
        if len(re.sub(r"\s+", " ", line).strip()) >= 20
    ]
    if not lines:
        return 0.0
    counts = Counter(lines)
    duplicate_instances = sum(count - 1 for count in counts.values() if count > 1)
    return duplicate_instances / max(len(lines), 1)


def private_use_ratio(text: str) -> float:
    if not text:
        return 0.0
    return sum(
        0xE000 <= ord(ch) <= 0xF8FF
        for ch in text
    ) / max(len(text), 1)


def token_diagnostics(text: str) -> Dict[str, float]:
    tokens = re.findall(r"\b[\wÀ-ÖØ-öø-ÿ]+\b", text, flags=re.UNICODE)
    if not tokens:
        return {
            "one_char_token_ratio": 1.0,
            "very_long_token_ratio": 1.0,
            "no_vowel_token_ratio": 1.0,
            "mixed_alnum_token_ratio": 0.0,
        }

    vowels = set("aeiouyäöüàâæçéèêëîïôœùûüÿ")
    one_char = sum(len(t) == 1 for t in tokens)
    very_long = sum(len(t) > 30 for t in tokens)
    no_vowel = sum(
        len(t) >= 5 and not any(ch.lower() in vowels for ch in t)
        for t in tokens
    )
    mixed = sum(any(ch.isalpha() for ch in t) and any(ch.isdigit() for ch in t) for t in tokens)

    n = len(tokens)
    return {
        "one_char_token_ratio": one_char / n,
        "very_long_token_ratio": very_long / n,
        "no_vowel_token_ratio": no_vowel / n,
        "mixed_alnum_token_ratio": mixed / n,
    }


def language_plausibility(text: str) -> Tuple[float, str]:
    words = re.findall(r"\b[\wÀ-ÖØ-öø-ÿ]+\b", text.lower(), flags=re.UNICODE)
    if not words:
        return 0.0, ""

    counts = Counter(words)
    scores = {}
    for lang in COMMON_WORDS:
        common_hits = sum(counts[w] for w in COMMON_WORDS[lang])
        legal_hits = sum(counts[w] for w in LEGAL_TERMS[lang])
        scores[lang] = min(1.0, (common_hits * 0.5 + legal_hits * 2.0) / max(len(words) * 0.01, 1))

    best_lang = max(scores, key=scores.get)
    return float(scores[best_lang]), best_lang


def text_quality_profile(raw_text: str, clean_text: str, meta: Dict[str, Any], file_format: str) -> Dict[str, Any]:
    raw_text = raw_text or ""
    clean_text = clean_text or ""
    n = len(clean_text)

    alpha_ratio = sum(ch.isalpha() for ch in clean_text) / max(n, 1)
    digit_ratio = sum(ch.isdigit() for ch in clean_text) / max(n, 1)
    printable_ratio = sum(ch.isprintable() or ch in "\n\t" for ch in clean_text) / max(n, 1)
    replacement_ratio = clean_text.count("�") / max(n, 1)
    mojibake_ratio = sum(clean_text.count(p) for p in MOJIBAKE_PATTERNS) / max(n, 1)
    control_ratio = sum(
        1 for ch in clean_text
        if unicodedata.category(ch).startswith("C") and ch not in "\n\t"
    ) / max(n, 1)

    token_stats = token_diagnostics(clean_text)
    lang_score, detected_language = language_plausibility(clean_text)
    repeat_ratio = repeated_character_ratio(clean_text)
    duplicate_ratio = duplicate_line_ratio(clean_text)
    pua_ratio = private_use_ratio(clean_text)

    length_component = min(25.0, n / 800.0)
    score = (
        length_component
        + printable_ratio * 15
        + min(alpha_ratio / 0.65, 1.0) * 10
        + lang_score * 20
        - replacement_ratio * 250
        - mojibake_ratio * 180
        - control_ratio * 250
        - pua_ratio * 250
        - repeat_ratio * 100
        - duplicate_ratio * 20
        - token_stats["one_char_token_ratio"] * 18
        - token_stats["very_long_token_ratio"] * 80
        - token_stats["no_vowel_token_ratio"] * 18
    )

    reasons = []
    if n < MIN_CHARS_ANY:
        reasons.append("very_short_text")
    elif file_format == "pdf" and n < MIN_PDF_CHARS_GOOD:
        reasons.append("short_pdf_text")
    if replacement_ratio > 0.002:
        reasons.append("replacement_characters")
    if mojibake_ratio > 0.0005:
        reasons.append("possible_mojibake")
    if pua_ratio > 0.0005:
        reasons.append("private_use_characters")
    if repeat_ratio > 0.01:
        reasons.append("repeated_character_garbage")
    if duplicate_ratio > 0.20:
        reasons.append("many_duplicate_lines")
    if token_stats["one_char_token_ratio"] > 0.30:
        reasons.append("many_one_character_tokens")
    if token_stats["very_long_token_ratio"] > 0.01:
        reasons.append("many_very_long_tokens")
    if lang_score < 0.10 and n > 1000:
        reasons.append("low_language_plausibility")

    if n < MIN_CHARS_ANY or score < MIN_SCORE_ACCEPTABLE:
        quality_flag = "failed"
    elif reasons:
        quality_flag = "fishy"
    else:
        quality_flag = "ok"

    return {
        "quality_score": round(float(score), 3),
        "quality_flag": quality_flag,
        "quality_reasons": "|".join(reasons),
        "n_chars_raw": len(raw_text),
        "n_chars_clean": n,
        "n_pages": meta.get("n_pages"),
        "alpha_ratio": round(alpha_ratio, 6),
        "digit_ratio": round(digit_ratio, 6),
        "printable_ratio": round(printable_ratio, 6),
        "replacement_ratio": round(replacement_ratio, 8),
        "mojibake_ratio": round(mojibake_ratio, 8),
        "private_use_ratio": round(pua_ratio, 8),
        "repeated_character_ratio": round(repeat_ratio, 8),
        "duplicate_line_ratio": round(duplicate_ratio, 8),
        "language_score": round(lang_score, 6),
        "detected_language": detected_language,
        **{k: round(v, 8) for k, v in token_stats.items()},
    }

## 4. HTML, TXT, native PDF, and MinerU extractors

In [4]:
HTML_REMOVE_SELECTORS = [
    "script", "style", "noscript", "nav", "footer", "header", "form", "aside",
    "[class*='cookie']", "[id*='cookie']", "[class*='share']",
    "[class*='language']", "[aria-label*='language']",
    "[class*='pagination']", "[class*='print']",
    "[class*='breadcrumb']", "[id*='breadcrumb']",
    "[class*='toolbar']", "[id*='toolbar']",
    "[class*='download']", "[id*='download']",
    "[class*='accessibility']", "[id*='accessibility']",
    "[class*='metadata']", "[id*='metadata']",
]

HTML_CANDIDATE_SELECTORS = [
    "main", "article", "#document1", "#TexteOnly", "#text",
    ".document-content", ".document", ".content", "body",
]


def html_container_score(tag) -> float:
    text = tag.get_text(" ", strip=True)
    if not text:
        return -100.0

    links = tag.find_all("a")
    link_text_chars = sum(len(a.get_text(" ", strip=True)) for a in links)
    link_density = link_text_chars / max(len(text), 1)
    paragraphs = len(tag.find_all(["p", "div", "li", "table"]))
    lang_score, _ = language_plausibility(text)

    return (
        min(len(text), 150_000) / 1500
        + min(paragraphs, 200) * 0.08
        + lang_score * 15
        - link_density * 40
    )


def extract_text_from_html(path: Path) -> Tuple[str, Dict[str, Any]]:
    data = read_file_bytes(path)
    html, decode_meta = decode_bytes_smart(data, is_html=True)

    if BeautifulSoup is None:
        text = re.sub(r"<script.*?</script>|<style.*?</style>", " ", html, flags=re.I | re.S)
        text = re.sub(r"<[^>]+>", " ", text)
        return normalize_unicode(text), {
            "method": "html_regex",
            "n_pages": None,
            **decode_meta,
        }

    soup = BeautifulSoup(html, "html.parser")

    for selector in HTML_REMOVE_SELECTORS:
        try:
            for tag in soup.select(selector):
                tag.decompose()
        except Exception:
            pass

    candidates = []
    seen_text_hashes = set()
    for selector in HTML_CANDIDATE_SELECTORS:
        try:
            for tag in soup.select(selector):
                text = tag.get_text("\n", strip=True)
                if len(text) < 100:
                    continue
                text_hash = hashlib.sha1(
                    re.sub(r"\s+", " ", text).encode("utf-8", errors="ignore")
                ).hexdigest()
                if text_hash in seen_text_hashes:
                    continue
                seen_text_hashes.add(text_hash)
                candidates.append((html_container_score(tag), selector, text))
        except Exception:
            pass

    if candidates:
        _, selected_selector, text = max(candidates, key=lambda x: x[0])
    else:
        selected_selector = "document"
        text = soup.get_text("\n", strip=True)

    return normalize_unicode(text), {
        "method": "html_bs4_best_container",
        "n_pages": None,
        "html_selected_container": selected_selector,
        "html_candidate_count": len(candidates),
        **decode_meta,
    }


def extract_text_from_plain(path: Path) -> Tuple[str, Dict[str, Any]]:
    text, decode_meta = decode_bytes_smart(read_file_bytes(path), is_html=False)
    return text, {"method": "plain_text_smart_decode", "n_pages": None, **decode_meta}


def extract_pdf_pymupdf_text(path: Path) -> Tuple[str, Dict[str, Any]]:
    if fitz is None:
        raise RuntimeError("PyMuPDF is not installed.")

    chunks = []
    with fitz.open(path) as doc:
        for page_no, page in enumerate(doc, start=1):
            text = page.get_text("text", sort=True) or ""
            chunks.append(f"\n\n[page {page_no}]\n{text}")
        n_pages = len(doc)

    return normalize_unicode("".join(chunks)), {
        "method": "pdf_pymupdf_text_sorted",
        "n_pages": n_pages,
    }


def extract_pdf_pymupdf_blocks(path: Path) -> Tuple[str, Dict[str, Any]]:
    if fitz is None:
        raise RuntimeError("PyMuPDF is not installed.")

    chunks = []
    with fitz.open(path) as doc:
        for page_no, page in enumerate(doc, start=1):
            blocks = page.get_text("blocks", sort=True) or []
            block_texts = []
            for block in blocks:
                if len(block) >= 5:
                    text = str(block[4] or "").strip()
                    if text:
                        block_texts.append(text)
            chunks.append(f"\n\n[page {page_no}]\n" + "\n\n".join(block_texts))
        n_pages = len(doc)

    return normalize_unicode("".join(chunks)), {
        "method": "pdf_pymupdf_blocks_sorted",
        "n_pages": n_pages,
    }


def extract_pdf_pdftotext(path: Path) -> Tuple[str, Dict[str, Any]]:
    if not PDFTOTEXT_CLI:
        raise RuntimeError("pdftotext is not installed or not on PATH.")

    with tempfile.TemporaryDirectory(prefix="pdftotext_") as tmpdir:
        out_txt = Path(tmpdir) / "output.txt"
        cmd = [PDFTOTEXT_CLI, "-layout", "-enc", "UTF-8", str(path), str(out_txt)]
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        if proc.returncode != 0:
            raise RuntimeError(proc.stderr[-2000:] or "pdftotext failed")
        text = out_txt.read_text(encoding="utf-8", errors="replace")

    n_pages = None
    if fitz is not None:
        try:
            with fitz.open(path) as doc:
                n_pages = len(doc)
        except Exception:
            pass

    return normalize_unicode(text), {
        "method": "pdf_pdftotext_layout",
        "n_pages": n_pages,
    }


def format_mineru_command(input_pdf: Path, output_dir: Path) -> List[str]:
    return [
        part.format(input_pdf=str(input_pdf), output_dir=str(output_dir))
        for part in MINERU_COMMAND_TEMPLATE
    ]


def run_mineru(input_pdf: Path, output_dir: Path) -> Dict[str, Any]:
    output_dir.mkdir(parents=True, exist_ok=True)
    cmd = format_mineru_command(input_pdf, output_dir)

    try:
        env = os.environ.copy()
        env.setdefault("ORT_LOG_SEVERITY_LEVEL", "3")
        proc = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=MINERU_TIMEOUT_SECONDS,
            env=env,
        )
        return {
            "ok": proc.returncode == 0,
            "returncode": proc.returncode,
            "cmd": " ".join(cmd),
            "stdout_tail": proc.stdout[-2000:],
            "stderr_tail": proc.stderr[-2000:],
        }
    except Exception as exc:
        return {
            "ok": False,
            "returncode": None,
            "cmd": " ".join(cmd),
            "error": repr(exc),
        }


def find_mineru_text(output_dir: Path) -> Optional[Path]:
    candidates: List[Path] = []
    for pattern in ["**/*.md", "**/*.txt"]:
        candidates.extend(output_dir.glob(pattern))

    candidates = [
        p for p in candidates
        if p.is_file()
        and p.stat().st_size > 0
        and not any(part.lower() in {"debug", "intermediate", "temp", "tmp"} for part in p.parts)
    ]
    if not candidates:
        return None

    expected_names = {"full.md", "content.md", "document.md", "result.md", "output.md"}

    def rank(path: Path) -> Tuple[int, int, int]:
        is_markdown = int(path.suffix.lower() == ".md")
        expected = int(path.name.lower() in expected_names)
        return expected, is_markdown, path.stat().st_size

    return max(candidates, key=rank)


def extract_pdf_mineru(path: Path) -> Tuple[str, Dict[str, Any]]:
    with tempfile.TemporaryDirectory(prefix="mineru_") as tmpdir:
        out_dir = Path(tmpdir)
        result = run_mineru(path, out_dir)
        if not result.get("ok"):
            raise RuntimeError(result.get("error") or result.get("stderr_tail") or "MinerU failed")

        text_file = find_mineru_text(out_dir)
        if text_file is None:
            raise RuntimeError("MinerU completed but produced no non-empty markdown/text file.")

        text = text_file.read_text(encoding="utf-8", errors="replace")
        source_markdown = text if text_file.suffix.lower() == ".md" else ""

    n_pages = None
    if fitz is not None:
        try:
            with fitz.open(path) as doc:
                n_pages = len(doc)
        except Exception:
            pass

    return normalize_unicode(text), {
        "method": "pdf_mineru",
        "n_pages": n_pages,
        "source_markdown": source_markdown,
        "mineru_output_name": text_file.name,
    }


def make_candidate(
    document_id: str,
    method: str,
    raw_text: str,
    meta: Dict[str, Any],
    file_format: str,
) -> Dict[str, Any]:
    readable_text = clean_text_readable(raw_text)
    regex_text = clean_text_for_regex_and_llm(raw_text)
    quality = text_quality_profile(raw_text, regex_text, meta, file_format)

    return {
        "document_id": document_id,
        "candidate_method": method,
        "raw_text": raw_text,
        "readable_text": readable_text,
        "clean_text": regex_text,
        "meta": meta,
        **quality,
    }



def generate_candidates(
    row: Dict[str, Any],
) -> Tuple[List[Dict[str, Any]], List[str]]:
    path = Path(str(row["raw_file_path"]))
    file_format = str(row.get("file_format", "")).lower()
    document_id = str(row["document_id"])
    candidates: List[Dict[str, Any]] = []
    errors: List[str] = []

    def attempt(method_name, extractor):
        try:
            raw_text, meta = extractor(path)
            candidates.append(
                make_candidate(
                    document_id=document_id,
                    method=method_name,
                    raw_text=raw_text,
                    meta=meta,
                    file_format=file_format,
                )
            )
        except Exception as exc:
            errors.append(f"{method_name}: {repr(exc)}")

    if file_format == "html":
        attempt("html_best_container", extract_text_from_html)

    elif file_format in {"txt", "text"}:
        attempt("plain_text_smart_decode", extract_text_from_plain)

    elif file_format == "pdf":
        attempt(
            "pdf_pymupdf_text_sorted",
            extract_pdf_pymupdf_text,
        )
        attempt(
            "pdf_pymupdf_blocks_sorted",
            extract_pdf_pymupdf_blocks,
        )

        if RUN_PDFTOTEXT and PDFTOTEXT_CLI:
            attempt(
                "pdf_pdftotext_layout",
                extract_pdf_pdftotext,
            )

        native_candidates = [
            candidate
            for candidate in candidates
            if candidate.get("candidate_method") != "pdf_mineru"
        ]
        best_native = max(
            native_candidates,
            key=candidate_preference,
            default=None,
        )

        best_native_score = (
            float(best_native.get("quality_score", -999))
            if best_native
            else -999
        )
        best_native_reasons = {
            reason
            for reason in str(
                best_native.get("quality_reasons", "")
                if best_native
                else ""
            ).split("|")
            if reason
        }

        should_run_ocr = (
            RUN_MINERU
            and MINERU_CLI is not None
            and (
                RUN_MINERU_FOR_ALL_PDFS
                or best_native is None
                or best_native.get("quality_flag") == "failed"
                or best_native_score < MIN_NATIVE_SCORE_TO_RUN_OCR
                or bool(best_native_reasons & SERIOUS_OCR_REASONS)
            )
        )

        if should_run_ocr:
            attempt("pdf_mineru", extract_pdf_mineru)

    else:
        errors.append(
            f"unsupported_file_format: {file_format}"
        )

    return candidates, errors

## 5. Candidate selection, outputs, and reuse checks

In [5]:
METHOD_PRIORITY = {
    "html_best_container": 6,
    "plain_text_smart_decode": 6,
    "pdf_pymupdf_text_sorted": 5,
    "pdf_pdftotext_layout": 4,
    "pdf_pymupdf_blocks_sorted": 3,
    "pdf_mineru": 1,
}


def candidate_preference(candidate: Dict[str, Any]) -> Tuple[float, int]:
    return (
        float(candidate.get("quality_score", -999)),
        METHOD_PRIORITY.get(str(candidate.get("candidate_method", "")), 0),
    )


def best_native_candidate(candidates: List[Dict[str, Any]]) -> Optional[Dict[str, Any]]:
    native = [
        candidate for candidate in candidates
        if candidate.get("candidate_method") != "pdf_mineru"
    ]
    if not native:
        return None

    best_score = max(float(candidate.get("quality_score", -999)) for candidate in native)
    near_best = [
        candidate for candidate in native
        if best_score - float(candidate.get("quality_score", -999)) <= METHOD_TIE_TOLERANCE
    ]
    return max(
        near_best,
        key=lambda candidate: METHOD_PRIORITY.get(
            str(candidate.get("candidate_method", "")), 0
        ),
    )


def quality_reason_set(candidate: Dict[str, Any]) -> set:
    return {
        reason for reason in str(candidate.get("quality_reasons", "")).split("|")
        if reason
    }


def select_best_candidate(
    candidates: List[Dict[str, Any]],
) -> Tuple[Optional[Dict[str, Any]], str]:
    if not candidates:
        return None, "no_candidates"

    best_native = best_native_candidate(candidates)
    mineru = next(
        (
            candidate for candidate in candidates
            if candidate.get("candidate_method") == "pdf_mineru"
        ),
        None,
    )

    if mineru is None:
        selected = best_native or max(candidates, key=candidate_preference)
        return selected, "best_available_candidate"

    if best_native is None:
        return mineru, "native_missing_mineru_available"

    native_score = float(best_native.get("quality_score", -999))
    mineru_score = float(mineru.get("quality_score", -999))
    native_length = int(best_native.get("n_chars_clean", 0) or 0)
    mineru_length = int(mineru.get("n_chars_clean", 0) or 0)

    native_serious = quality_reason_set(best_native) & SERIOUS_OCR_REASONS
    mineru_serious = quality_reason_set(mineru) & SERIOUS_OCR_REASONS

    if (
        best_native.get("quality_flag") == "failed"
        and mineru.get("quality_flag") != "failed"
    ):
        return mineru, "native_failed_mineru_valid"

    gain = mineru_score - native_score
    length_ok = (
        native_length <= 0
        or mineru_length >= MIN_MINERU_NATIVE_LENGTH_RATIO * native_length
        or bool(native_serious)
    )
    serious_ok = not (mineru_serious - native_serious)

    if gain >= MIN_MINERU_SCORE_GAIN and length_ok and serious_ok:
        return mineru, f"mineru_gain_{gain:.3f}_meets_threshold"

    reasons = [
        f"mineru_gain_{gain:.3f}_below_or_blocked",
        f"length_ok={length_ok}",
        f"serious_ok={serious_ok}",
    ]
    return best_native, ";".join(reasons)


def output_paths_for(document_id: str) -> Tuple[Path, Path, Path]:
    safe_document_id = safe_id(document_id)
    return (
        TEXT_DIR / f"{safe_document_id}.txt",
        TEXT_DIR / f"{safe_document_id}__readable.txt",
        MARKDOWN_DIR / f"{safe_document_id}.md",
    )


def save_candidate_files(
    document_id: str,
    candidate: Dict[str, Any],
) -> Dict[str, str]:
    method = safe_id(candidate["candidate_method"])
    folder = CANDIDATE_DATA_DIR / safe_id(document_id)
    folder.mkdir(parents=True, exist_ok=True)

    raw_path = folder / f"{method}__raw.txt"
    readable_path = folder / f"{method}__readable.txt"
    regex_path = folder / f"{method}__regex.txt"

    raw_path.write_text(candidate.get("raw_text", ""), encoding="utf-8")
    readable_path.write_text(candidate.get("readable_text", ""), encoding="utf-8")
    regex_path.write_text(candidate.get("clean_text", ""), encoding="utf-8")

    return {
        "candidate_raw_path": str(raw_path),
        "candidate_readable_path": str(readable_path),
        "candidate_regex_path": str(regex_path),
    }


def markdown_header(row: Dict[str, Any], selected: Dict[str, Any]) -> str:
    meta = {
        "document_id": row.get("document_id"),
        "source": "court_infocuria",
        "document_key": row.get("document_key"),
        "case_number": row.get("case_number"),
        "title": row.get("case_name"),
        "document_type": row.get("document_type"),
        "document_date": row.get("document_date"),
        "language": row.get("language"),
        "source_url": row.get("source_url"),
        "file_format": row.get("file_format"),
        "raw_file_path": row.get("raw_file_path"),
        "extraction_version": EXTRACTION_VERSION,
        "selected_method": selected.get("candidate_method"),
        "quality_score": selected.get("quality_score"),
        "quality_flag": selected.get("quality_flag"),
        "quality_reasons": selected.get("quality_reasons"),
        "created_at": datetime.now().isoformat(timespec="seconds"),
    }
    return (
        "---\n"
        + "\n".join(
            f"{key}: {json.dumps(value, ensure_ascii=False)}"
            for key, value in meta.items()
        )
        + "\n---\n\n"
    )


def write_markdown(
    row: Dict[str, Any],
    selected: Dict[str, Any],
    markdown_path: Path,
) -> None:
    header = markdown_header(row, selected)
    source_markdown = str(selected.get("meta", {}).get("source_markdown", "") or "")
    body = source_markdown if (
        selected.get("candidate_method") == "pdf_mineru" and source_markdown.strip()
    ) else selected.get("readable_text", "")
    markdown_path.write_text(header + body, encoding="utf-8")


def load_existing_manifests():
    old_clean = pd.DataFrame()
    old_candidates = pd.DataFrame()

    if CLEAN_FILE_MANIFEST_PATH.exists() and CLEAN_FILE_MANIFEST_PATH.stat().st_size > 0:
        try:
            old_clean = pd.read_csv(CLEAN_FILE_MANIFEST_PATH, low_memory=False)
        except pd.errors.EmptyDataError:
            pass

    if CANDIDATE_MANIFEST_PATH.exists() and CANDIDATE_MANIFEST_PATH.stat().st_size > 0:
        try:
            old_candidates = pd.read_csv(CANDIDATE_MANIFEST_PATH, low_memory=False)
        except pd.errors.EmptyDataError:
            pass

    clean_lookup = {
        str(row["document_id"]): row.to_dict()
        for _, row in old_clean.iterrows()
        if "document_id" in old_clean.columns
    }
    return old_clean, old_candidates, clean_lookup


previous_clean_manifest, previous_candidate_manifest, previous_clean_lookup = (
    load_existing_manifests()
)


def existing_output_is_current(row: Dict[str, Any]) -> bool:
    if FORCE_REPROCESS:
        return False

    previous = previous_clean_lookup.get(str(row.get("document_id", "")))
    if not previous:
        return False

    try:
        version_ok = int(float(previous.get("extraction_version", 0))) == EXTRACTION_VERSION
    except Exception:
        version_ok = False

    required = [
        previous.get("clean_text_path", ""),
        previous.get("readable_text_path", ""),
        previous.get("markdown_path", ""),
    ]
    paths_present = all(str(path).strip() for path in required)
    paths_ok = paths_present and all(
        Path(str(path)).exists() and Path(str(path)).stat().st_size > 0
        for path in required
    )

    source_size_ok = str(previous.get("source_size_bytes", "")) == str(row.get("source_size_bytes", ""))
    source_mtime_ok = str(previous.get("source_modified_ns", "")) == str(row.get("source_modified_ns", ""))

    return version_ok and paths_ok and source_size_ok and source_mtime_ok


def candidate_manifest_record(
    row: Dict[str, Any],
    candidate: Dict[str, Any],
    selected: Dict[str, Any],
    selection_reason: str,
) -> Dict[str, Any]:
    meta = candidate.get("meta", {}) or {}
    record = {
        "source_row_i": row.get("source_row_i"),
        "document_id": row.get("document_id"),
        "case_number": row.get("case_number"),
        "document_type": row.get("document_type"),
        "document_date": row.get("document_date"),
        "language": row.get("language"),
        "raw_file_path": row.get("raw_file_path"),
        "file_format": row.get("file_format"),
        "candidate_method": candidate.get("candidate_method"),
        "candidate_selected": candidate is selected,
        "selection_reason": selection_reason if candidate is selected else "",
        "quality_score": candidate.get("quality_score"),
        "quality_flag": candidate.get("quality_flag"),
        "quality_reasons": candidate.get("quality_reasons"),
        "n_chars_raw": candidate.get("n_chars_raw"),
        "n_chars_clean": candidate.get("n_chars_clean"),
        "n_words_clean": candidate.get("n_words_clean"),
        "detected_language": candidate.get("detected_language"),
        "n_pages": meta.get("n_pages"),
        "encoding": meta.get("encoding"),
        "encoding_source": meta.get("encoding_source"),
        "html_selected_container": meta.get("html_selected_container"),
        "mineru_output_name": meta.get("mineru_output_name"),
        "extraction_version": EXTRACTION_VERSION,
    }

    should_save = (
        SAVE_ALL_CANDIDATE_TEXTS
        or candidate is selected
        or (
            SAVE_REVIEW_CANDIDATE_TEXTS
            and candidate.get("quality_flag") in {"fishy", "failed", "check"}
        )
    )
    if should_save:
        record.update(save_candidate_files(str(row["document_id"]), candidate))
    else:
        record.update({
            "candidate_raw_path": "",
            "candidate_readable_path": "",
            "candidate_regex_path": "",
        })
    return record

## 6. Process files and write InfoCuria CSV/XLSX manifests

In [6]:
def empty_clean_record(row: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "source_row_i": row.get("source_row_i", ""),
        "case_number": row.get("case_number", ""),
        "document_id": row.get("document_id", ""),
        "document_key": row.get("document_key", ""),
        "document_type": row.get("document_type", ""),
        "document_date": row.get("document_date", ""),
        "language": row.get("language", ""),
        "source_url": row.get("source_url", ""),
        "downloaded_file_path": row.get("downloaded_file_path", ""),
        "raw_file_path": row.get("raw_file_path", ""),
        "file_exists": row.get("file_exists", False),
        "file_type": row.get("file_type", ""),
        "file_format": row.get("file_format", ""),
        "source_size_bytes": row.get("source_size_bytes"),
        "source_modified_ns": row.get("source_modified_ns"),
        "extraction_version": EXTRACTION_VERSION,
        "cleaning_success": False,
        "cleaning_status": "",
        "selected_method": "",
        "extraction_method": "",
        "selection_reason": "",
        "quality_score": None,
        "quality_flag": "",
        "quality_reasons": "",
        "quality_reason": "",
        "n_chars_clean": 0,
        "n_words_clean": 0,
        "n_pages": None,
        "native_best_method": "",
        "native_best_score": None,
        "mineru_attempted": False,
        "mineru_selected": False,
        "mineru_score": None,
        "mineru_score_gain": None,
        "extraction_errors": "",
        "clean_text_path": "",
        "readable_text_path": "",
        "markdown_path": "",
        "clean_markdown_path": "",
        "cleaning_error": "",
        "processed_at": datetime.now().isoformat(timespec="seconds"),
    }


def process_one(
    row: Dict[str, Any],
) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    out = empty_clean_record(row)

    if not row.get("processing_eligible", False):
        out.update({
            "cleaning_status": "skipped",
            "quality_flag": "skipped",
            "quality_reasons": row.get("processing_skip_reason", ""),
            "quality_reason": row.get("processing_skip_reason", ""),
        })
        return out, []

    clean_path, readable_path, markdown_path = output_paths_for(str(row["document_id"]))
    out.update({
        "clean_text_path": str(clean_path),
        "readable_text_path": str(readable_path),
        "markdown_path": str(markdown_path),
        "clean_markdown_path": str(markdown_path),
    })

    if existing_output_is_current(row):
        previous = previous_clean_lookup[str(row["document_id"])].copy()
        previous["cleaning_status"] = "skipped_current_v4"
        previous["processed_at"] = datetime.now().isoformat(timespec="seconds")
        return previous, []

    try:
        candidates, extraction_errors = generate_candidates(row)
        selected, selection_reason = select_best_candidate(candidates)

        if selected is None:
            out.update({
                "cleaning_status": "failed_no_candidate",
                "quality_flag": "failed",
                "quality_reasons": "no_candidate",
                "quality_reason": "no_candidate",
                "extraction_errors": " | ".join(extraction_errors),
            })
            return out, []

        native = best_native_candidate(candidates)
        mineru = next(
            (
                candidate for candidate in candidates
                if candidate.get("candidate_method") == "pdf_mineru"
            ),
            None,
        )

        clean_path.write_text(selected.get("clean_text", ""), encoding="utf-8")
        readable_path.write_text(selected.get("readable_text", ""), encoding="utf-8")
        write_markdown(row, selected, markdown_path)

        selected_meta = selected.get("meta", {}) or {}
        native_score = float(native.get("quality_score")) if native else None
        mineru_score = float(mineru.get("quality_score")) if mineru else None

        out.update({
            "cleaning_success": bool(str(selected.get("clean_text", "")).strip()),
            "cleaning_status": (
                "processed"
                if str(selected.get("clean_text", "")).strip()
                else "failed_empty_output"
            ),
            "selected_method": selected.get("candidate_method", ""),
            "extraction_method": selected.get("candidate_method", ""),
            "selection_reason": selection_reason,
            "quality_score": selected.get("quality_score"),
            "quality_flag": selected.get("quality_flag"),
            "quality_reasons": selected.get("quality_reasons", ""),
            "quality_reason": selected.get("quality_reasons", ""),
            "n_chars_clean": selected.get("n_chars_clean", 0),
            "n_words_clean": selected.get("n_words_clean", 0),
            "n_pages": selected_meta.get("n_pages"),
            "native_best_method": native.get("candidate_method", "") if native else "",
            "native_best_score": native_score,
            "mineru_attempted": mineru is not None,
            "mineru_selected": selected.get("candidate_method") == "pdf_mineru",
            "mineru_score": mineru_score,
            "mineru_score_gain": (
                mineru_score - native_score
                if mineru_score is not None and native_score is not None
                else None
            ),
            "extraction_errors": " | ".join(extraction_errors),
        })

        candidate_records = [
            candidate_manifest_record(row, candidate, selected, selection_reason)
            for candidate in candidates
        ]
        return out, candidate_records

    except Exception as exc:
        out.update({
            "cleaning_status": "failed_exception",
            "quality_flag": "failed",
            "quality_reasons": "exception",
            "quality_reason": "exception",
            "cleaning_error": repr(exc),
        })
        error_path = ERROR_DIR / f"{safe_id(str(row.get('document_id', 'missing')))}__error.txt"
        error_path.write_text(repr(exc), encoding="utf-8")
        return out, []


to_process = work_df.copy()
if PROCESS_LIMIT is not None:
    to_process = to_process.head(int(PROCESS_LIMIT)).copy()

clean_records: List[Dict[str, Any]] = []
candidate_records: List[Dict[str, Any]] = []

for _, manifest_row in tqdm(
    to_process.iterrows(),
    total=len(to_process),
    desc="Processing Court InfoCuria files",
):
    clean_record, document_candidates = process_one(manifest_row.to_dict())
    clean_records.append(clean_record)
    candidate_records.extend(document_candidates)

    if SAVE_EVERY and len(clean_records) % SAVE_EVERY == 0:
        pd.DataFrame(clean_records).to_csv(CLEAN_FILE_MANIFEST_PATH, index=False)
        pd.DataFrame(candidate_records).to_csv(CANDIDATE_MANIFEST_PATH, index=False)

clean_df = pd.DataFrame(clean_records)
candidate_df = pd.DataFrame(candidate_records)

clean_df.to_csv(CLEAN_FILE_MANIFEST_PATH, index=False)
clean_df.to_excel(CLEAN_FILE_MANIFEST_XLSX, index=False)

candidate_df.to_csv(CANDIDATE_MANIFEST_PATH, index=False)
candidate_df.to_excel(CANDIDATE_MANIFEST_XLSX, index=False)

print("Wrote:", CLEAN_FILE_MANIFEST_PATH)
print("Wrote:", CLEAN_FILE_MANIFEST_XLSX)
print("Wrote:", CANDIDATE_MANIFEST_PATH)
print("Wrote:", CANDIDATE_MANIFEST_XLSX)
print("Rows:", len(clean_df))
print("Cleaning success:", int(clean_df["cleaning_success"].fillna(False).sum()))
print(clean_df["cleaning_status"].value_counts(dropna=False))
print("\nSelected methods:")
print(clean_df["selected_method"].value_counts(dropna=False))
display(clean_df.head(20))

Processing Court InfoCuria files:   0%|          | 0/77085 [00:00<?, ?it/s]

Wrote: /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_clean_file_manifest.csv
Wrote: /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_clean_file_manifest.xlsx
Wrote: /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_extraction_candidate_manifest.csv
Wrote: /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_extraction_candidate_manifest.xlsx
Rows: 77085
Cleaning success: 77085
cleaning_status
processed    77085
Name: count, dtype: int64

Selected methods:
selected_method
html_best_container          62280
pdf_pymupdf_text_sorted      14635
pdf_pdftotext_layout           120
pdf_pymupdf_blocks_sorted       50
Name: count, dtype: int64


,source_row_i,case_number,document_id,document_key,document_type,document_date,language,source_url,downloaded_file_path,raw_file_path,...,mineru_selected,mineru_score,mineru_score_gain,extraction_errors,clean_text_path,readable_text_path,markdown_path,clean_markdown_path,cleaning_error,processed_at
0,0,C-1/00,1216021,id_85906,ARRET_SOM,2001-12-13,EN,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/projects/eccjeu/data/raw/court_info...,...,False,None,None,,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,,2026-07-17T07:47:06
1,1,C-1/00,616977,id_46950,ARRET,2001-12-13,EN,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/projects/eccjeu/data/raw/court_info...,...,False,None,None,,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,,2026-07-17T07:47:06
2,2,C-1/00 SA,616472,id_46421,ORD,2001-05-29,EN,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/projects/eccjeu/data/raw/court_info...,...,False,None,None,,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,,2026-07-17T07:47:07
3,3,C-1/00 SA,1216138,id_85947,ORD_SOM,2001-05-29,EN,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/projects/eccjeu/data/raw/court_info...,...,False,None,None,,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,,2026-07-17T07:47:07
4,4,C-1/01 P,1258909,id_85572,ORD_SOM,2001-09-20,EN,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/projects/eccjeu/data/raw/court_info...,...,False,None,None,,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,,2026-07-17T07:47:07
5,5,C-1/01 P,616689,id_46652,ORD,2001-09-20,EN,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/projects/eccjeu/data/raw/court_info...,...,False,None,None,,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,,2026-07-17T07:47:07
6,6,C-1/02,618967,id_49061,ARRET,2004-04-01,EN,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/projects/eccjeu/data/raw/court_info...,...,False,None,None,,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,,2026-07-17T07:47:07
7,7,C-1/02,701344,id_84842,ORD_COMM,2003-08-22,EN,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/projects/eccjeu/data/raw/court_info...,...,False,None,None,,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/processed/cour...,,2026-07-17T07:47:07
8,8,C-1/02,624317,id_54478,ORD_COMM,2003-09-06,FR,https://infocuriaws.curia.europa.eu/blob/downl...,/home/edik/projects/eccjeu/data/raw/court_info...,/home/edik/project

## 7. Inspect failures and review candidates

In [7]:
if CLEAN_FILE_MANIFEST_PATH.exists():
    review_df = pd.read_csv(CLEAN_FILE_MANIFEST_PATH, low_memory=False)
    problem = review_df[
        (review_df["cleaning_success"] != True)
        | (review_df["quality_flag"].isin(["fishy", "failed", "check", "skipped"]))
    ].copy()

    print("Problem/check rows:", len(problem))
    columns = [
        "case_number", "document_id", "document_type", "document_date", "language",
        "raw_file_path", "file_format", "selected_method", "selection_reason",
        "mineru_attempted", "mineru_selected", "quality_score", "quality_flag",
        "quality_reasons", "n_chars_clean", "extraction_errors", "cleaning_error",
    ]
    display(problem[[column for column in columns if column in problem.columns]].head(100))
else:
    print("Run the processing cell first.")

Problem/check rows: 3250


,case_number,document_id,document_type,document_date,language,raw_file_path,file_format,selected_method,selection_reason,mineru_attempted,mineru_selected,quality_score,quality_flag,quality_reasons,n_chars_clean,extraction_errors,cleaning_error
1,C-1/00,616977,ARRET,2001-12-13,EN,/home/edik/projects/eccjeu/data/raw/court_info...,html,html_best_container,best_available_candidate,False,False,69.137,fishy,possible_mojibake,71511,NaN,NaN
7,C-1/02,701344,ORD_COMM,2003-08-22,EN,/home/edik/projects/eccjeu/data/raw/court_info...,html,html_best_container,best_available_candidate,False,False,42.149,fishy,repeated_character_garbage,1019,NaN,NaN
8,C-1/02,624317,ORD_COMM,2003-09-06,FR,/home/edik/projects/eccjeu/data/raw/court_info...,html,html_best_container,best_available_candidate,False,False,42.037,fishy,repeated_character_garbage,1095,NaN,NaN
16,C-1/04,705940,ORD_COMM,2005-03-04,EN,/home/edik/projects/eccjeu/data/raw/court_info...,html,html_best_container,best_available_candidate,False,False,43.055,fishy,repeated_character_garbage,1158,NaN,NaN
22,C-1/05 SA,627193,ORD_COMM,2006-01-28,EN,/home/edik/projects/eccjeu/data/raw/court_info...,html,html_best_container,best_available_candidate,False,False,42.339,fishy,repeated_character_garbage,969,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3475,C-34/02,617895,ARRET,2003-06-19,EN,/home/edik/projects/eccjeu/data/raw/court_info...,html,html_best_container,best_available_candidate,False,False,69.226,fishy,possible_mojibake,43282,NaN,NaN
3496,C-34/12 P,1778949,ORD_COMM,2013-11-08,EN,/home/edik/projects/eccjeu/data/raw/court_info...,html,html_best_container,best_available_candidate,False,False,43.960,fishy,repeated_character_garbage,1620,NaN,NaN
3517,C-34/23,3157773,ORD_COMM,2024-05-17,EN,/home/edik/projects/eccjeu/data/raw/court_info...,html,html_best_container,best_available_candidate,False,False,42.484,fishy,repeated_character_garbage,907,NaN,NaN
3565,C-35/00,617066,ARRET,2002-01-24,EN,/home/edik/projects/eccjeu/data/raw/court_info...,html,html_best_container,best_available_candidate,False,False,61.294,fishy,possible_mojibake,13862,NaN,NaN


## 8. Preview one processed document

In [8]:
CASE_TO_VIEW = None  # e.g. "C-11/71"
ROW_TO_VIEW = None   # e.g. 0

if CLEAN_FILE_MANIFEST_PATH.exists():
    preview_df = pd.read_csv(CLEAN_FILE_MANIFEST_PATH, low_memory=False)

    if CASE_TO_VIEW:
        hit = preview_df[
            preview_df["case_number"].astype(str).str.contains(
                str(CASE_TO_VIEW), regex=False, na=False
            )
        ]
    elif ROW_TO_VIEW is not None:
        hit = preview_df.iloc[[int(ROW_TO_VIEW)]]
    else:
        hit = preview_df[preview_df["cleaning_success"] == True].head(1)

    if hit.empty:
        print("No matching processed document found.")
    else:
        record = hit.iloc[0].to_dict()
        keys = [
            "case_number", "document_id", "document_type", "document_date",
            "language", "raw_file_path", "file_format", "selected_method",
            "selection_reason", "quality_score", "quality_flag", "n_chars_clean",
            "clean_text_path", "readable_text_path", "markdown_path",
        ]
        print(json.dumps({key: record.get(key, "") for key in keys}, indent=2, ensure_ascii=False))

        text_path = Path(str(record["clean_text_path"]))
        print("\n--- REGEX / LLM TEXT SAMPLE ---\n")
        print(text_path.read_text(encoding="utf-8", errors="replace")[:5000])
else:
    print("Run the processing cell first.")

{
  "case_number": "C-1/00",
  "document_id": "1216021",
  "document_type": "ARRET_SOM",
  "document_date": "2001-12-13",
  "language": "EN",
  "raw_file_path": "/home/edik/projects/eccjeu/data/raw/court_infocuria/pdfs/85906-EN-1.pdf",
  "file_format": "pdf",
  "selected_method": "pdf_pymupdf_text_sorted",
  "selection_reason": "best_available_candidate",
  "quality_score": 50.832,
  "quality_flag": "ok",
  "n_chars_clean": 5447,
  "clean_text_path": "/home/edik/projects/eccjeu/data/processed/court_infocuria/1216021.txt",
  "readable_text_path": "/home/edik/projects/eccjeu/data/processed/court_infocuria/1216021__readable.txt",
  "markdown_path": "/home/edik/projects/eccjeu/data/processed/court_infocuria/1216021.md"
}

--- REGEX / LLM TEXT SAMPLE ---

[page 1] Case C-1/00

Commission of the European Communities v French Republic

(Failure by a Member State to fulfil its obligations — Refusal to end the ban on British beef and veal)

Opinion of Advocate General Mischo delivered on 20 Sep